# RTX A2000 hardware experiments — standalone

Notebook này chỉ chạy thí nghiệm phần cứng. Nó không tải dataset, không train và không chạy ISIC/OOD/CIFAR.

Pipeline: kiểm tra môi trường → kiểm tra bốn checkpoint fair-v3 → Level 1–2 structural profile → ONNX preflight → TensorRT Level 3 → đọc quality gates và bảng LaTeX.

## 0. Cấu hình local

Mở notebook từ root repository sau khi đã `git pull`. Sửa `TRTEXEC` nếu TensorRT nằm ở thư mục khác.

In [ ]:
import os, sys, json, shutil, subprocess, shlex
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
assert (REPO_ROOT / "experiments/nvidia_sparse_benchmark.py").exists(), (
    "Hãy mở notebook từ root repository MDEP."
)
os.chdir(REPO_ROOT)

TRTEXEC = Path(os.environ.get(
    "TRTEXEC_PATH",
    r"D:\Dependencies\TensorRT-11.1.0.106\bin\trtexec.exe",
))
SEED = 42
BATCH_SIZES = [1, 64]
REPEATS = 10
DURATION_SECONDS = 30
WARMUP_MS = 2000
RUN_LEVEL12 = True
RUN_EXPORT_PREFLIGHT = True
RUN_TENSORRT_LEVEL3 = True
PROTOCOL = "isic_fair_v3_nvidia24_2026_07_09"
OUTPUT_ROOT = REPO_ROOT / "paper_experiment_outputs/hardware_nvidia_rtx_a2000"

def run(cmd, check=True):
    cmd = [str(x) for x in cmd]
    print("\n$", shlex.join(cmd), flush=True)
    return subprocess.run(cmd, cwd=REPO_ROOT, check=check)

print("Repository:", REPO_ROOT)
print("Python:", sys.executable)
print("trtexec:", TRTEXEC)


## 1. Kiểm tra RTX A2000, CUDA và TensorRT

In [ ]:
import torch

assert torch.cuda.is_available(), "PyTorch không nhận CUDA."
gpu_name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
assert "RTX A2000" in gpu_name.upper(), f"GPU hiện tại không phải RTX A2000: {gpu_name}"
assert capability[0] >= 8, f"Compute capability không hỗ trợ Ampere sparse path: {capability}"
assert TRTEXEC.exists(), f"Không tìm thấy trtexec.exe: {TRTEXEC}"

run(["nvidia-smi"])
run([TRTEXEC, "--version"])
print({
    "gpu": gpu_name,
    "compute_capability": capability,
    "torch": torch.__version__,
    "cuda_runtime": torch.version.cuda,
})


## 1.5. Huấn luyện các mô hình phần cứng cục bộ (Tùy chọn)

Nếu bạn muốn huấn luyện trực tiếp trên GPU máy này thay vì tải từ Kaggle về, hãy đặt `RUN_TRAINING = True` ở cell dưới. 
Quá trình này sẽ chạy huấn luyện 40 epochs cho 4 mô hình (`dense_edl`, `static_24_edl`, `rigl_style_24`, và `full_guds`) dưới seed 42 và tự động lưu checkpoints.

In [ ]:
import sys
RUN_TRAINING = True

MODELS_TO_TRAIN = ["dense_edl", "static_24_edl", "rigl_style_24", "full_guds"]
missing_any = False
for name in MODELS_TO_TRAIN:
    checkpoint = REPO_ROOT / "paper_experiment_outputs/isic" / f"{name}_fair_v3_nvidia24" / f"seed_{SEED}" / "model_state.pth"
    if not checkpoint.exists():
        missing_any = True
        break

if missing_any and not RUN_TRAINING:
    print("[WARN] Thiếu checkpoint cục bộ. Nếu máy bạn có GPU đủ mạnh để train, hãy đặt RUN_TRAINING = True để tự động huấn luyện.")
    print("Nếu không, hãy tải các checkpoint đã train từ Kaggle về và giải nén vào thư mục dự án.")

if RUN_TRAINING:
    print("=== Bắt đầu huấn luyện các mô hình phần cứng cục bộ ===")
    env_override = {
        "MDEP_DETERMINISTIC": "1",
        "PYTHONHASHSEED": "0",
        "CUBLAS_WORKSPACE_CONFIG": ":4096:8",
    }
    
    for name in MODELS_TO_TRAIN:
        checkpoint = REPO_ROOT / "paper_experiment_outputs/isic" / f"{name}_fair_v3_nvidia24" / f"seed_{SEED}" / "model_state.pth"
        if checkpoint.exists():
            print(f"[SKIP] {name} đã có checkpoint cục bộ.")
            continue
            
        print(f"\n--- Đang huấn luyện {name} (40 epochs) ---")
        cmd = [
            sys.executable, "-u", "experiments/isic_paper_experiments.py",
            "--experiment", name,
            "--epochs", "40",
            "--batch_size", "32",
            "--lr", "4e-5",
            "--seed", str(SEED),
            "--split_seed", "42",
            "--subsample_scope", "train",
            "--subsample_ratio", "20",
            "--structural_proxy_batches", "4",
            "--checkpoint_selection", "last",
            "--run_suffix", "_fair_v3_nvidia24"
        ]
        # Use the wrapper run function defined in Cell 0
        run(cmd)
    print("=== Hoàn tất huấn luyện cục bộ! ===")

## 2. Kiểm tra bốn checkpoint đầu vào

Checkpoint phải nằm đúng cấu trúc dưới `paper_experiment_outputs/isic/` và có `metrics.json` cùng thư mục.

In [ ]:
MODELS = ["dense_edl", "static_24_edl", "rigl_style_24", "full_guds"]
CHECKPOINTS = {
    name: REPO_ROOT / "paper_experiment_outputs/isic" / f"{name}_fair_v3_nvidia24" / f"seed_{SEED}" / "model_state.pth"
    for name in MODELS
}

for name, checkpoint in CHECKPOINTS.items():
    metrics_path = checkpoint.parent / "metrics.json"
    assert checkpoint.exists(), f"Thiếu checkpoint {name}: {checkpoint}"
    assert metrics_path.exists(), f"Thiếu metrics.json cho {name}: {metrics_path}"
    metadata = json.loads(metrics_path.read_text(encoding="utf-8"))
    assert metadata.get("protocol_version") == PROTOCOL, (
        f"Sai protocol cho {name}: {metadata.get('protocol_version')}"
    )
    print(f"[OK] {name}: {checkpoint}")


## 3. Level 1–2: structural feasibility và masked-PyTorch diagnostic

Phần này không phải sparse-kernel speedup. Nó chỉ báo density, exact 2:4, MAC/FLOP, masked throughput và CUDA memory.

In [ ]:
if RUN_LEVEL12:
    run([
        sys.executable, "-u", "experiments/hardware_profile.py",
        "--modes", "dense", "static_24", "guds",
        "--batch_size", "64", "--image_size", "224",
        "--warmup", "50", "--iters", "200", "--no_pretrained",
    ])
else:
    print("[SKIP] RUN_LEVEL12=False")


## 4. ONNX export preflight

Bước này kiểm tra checkpoint, NVIDIA-layout 2:4, frozen-graph equivalence và ONNX checker trước khi build TensorRT.

In [ ]:
if RUN_EXPORT_PREFLIGHT:
    run([
        sys.executable, "-u", "experiments/nvidia_sparse_benchmark.py",
        "--seed", str(SEED), "--export_only",
    ])
else:
    print("[SKIP] RUN_EXPORT_PREFLIGHT=False")


## 5. TensorRT Level 3

Runner tách network comparison khỏi same-checkpoint kernel ablation, randomize thứ tự chạy và chỉ sinh bảng paper khi toàn bộ quality gates đạt.

In [ ]:
if RUN_TENSORRT_LEVEL3:
    command = [
        sys.executable, "-u", "experiments/nvidia_sparse_benchmark.py",
        "--trtexec", TRTEXEC,
        "--seed", str(SEED),
        "--repeats", str(REPEATS),
        "--duration_s", str(DURATION_SECONDS),
        "--warmup_ms", str(WARMUP_MS),
        "--batch_sizes", *map(str, BATCH_SIZES),
    ]
    run(command)
else:
    print("[SKIP] RUN_TENSORRT_LEVEL3=False")


## 6. Đọc quality gates và kết quả mới nhất

In [ ]:
import pandas as pd

result_files = sorted(OUTPUT_ROOT.glob(f"seed_{SEED}_*/results.json"), key=lambda p: p.stat().st_mtime)
assert result_files, f"Không tìm thấy results.json trong {OUTPUT_ROOT}"
latest_result = result_files[-1]
result = json.loads(latest_result.read_text(encoding="utf-8"))

print("Result:", latest_result)
print("paper_ready:", result.get("paper_ready"))
display(pd.DataFrame([result.get("quality_gates", {})]).T.rename(columns={0: "passed"}))

summary_path = latest_result.parent / "summary.csv"
summary = pd.read_csv(summary_path)
display(summary.sort_values(["comparison", "batch_size", "model", "sparsity_enabled"]))

if result.get("paper_ready"):
    print("[PAPER READY] LaTeX table:", latest_result.parent / "paper_table.tex")
else:
    print("[NOT PAPER READY] Kiểm tra quality gates và logs/build_*_sparse.log; không đưa số vào bài.")


## Output cần giữ

- `results.json`: manifest và quality gates.
- `summary.csv`: kết quả tổng hợp.
- `raw_repeats.csv`: từng lượt đo.
- `logs/build_*_sparse.log`: bằng chứng TensorRT chọn sparse tactics.
- `paper_table.tex`: chỉ sinh khi `paper_ready=true`.

Không commit engine, ONNX, checkpoint hoặc log benchmark lên GitHub.